# Nyaya — does the fine-tuned embedder move the reader?

`nyaya-embed-v1` lifts full-hit recall@8 on the never-audited slice from 81.4% (BM25) to
88.1% (BM25 + embed-v1, RRF). Retrieval decides answers here (63% fact recall when the gold
section is in context, 20% when it is not), so this run holds the reader fixed — base
Qwen2.5-3B-Instruct, 768 new tokens, k=8 — and swaps only the dense model. The paired
bootstrap against the committed `base-768` predictions (same 413 questions, zero-shot
e5-base) is the claim test.

**Settings:** GPU T4 x2, Internet On, Input: `jitendrajha98/nyaya-model-src`. ~1.5 h.


In [ ]:
# --- setup -------------------------------------------------------------
import glob, os, shutil, subprocess, sys, time

os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["WANDB_DISABLED"] = "true"

cands = glob.glob("/kaggle/input/nyaya-model-src/**/pyproject.toml", recursive=True)
if not cands:
    import kagglehub
    root = kagglehub.dataset_download("jitendrajha98/nyaya-model-src")
    cands = glob.glob(os.path.join(root, "**", "pyproject.toml"), recursive=True)
assert cands, "dataset jitendrajha98/nyaya-model-src not available to this kernel"
SRC = os.path.dirname(cands[0])
WORK = "/kaggle/working/nyaya-model"
shutil.copytree(SRC, WORK, dirs_exist_ok=True)
os.chdir(WORK)
sys.path.insert(0, "src")

PROGRESS = "/kaggle/working/progress.txt"


def note(msg: str) -> None:
    print(msg, flush=True)
    with open(PROGRESS, "a", encoding="utf-8") as fh:
        fh.write(time.strftime("%H:%M:%S ") + msg + chr(10))


def run(cmd):
    note("$ " + " ".join(cmd))
    proc = subprocess.run(cmd, text=True)
    if proc.returncode != 0:
        note(f"FAILED exit {proc.returncode}")
        raise RuntimeError(f"step failed (exit {proc.returncode}): {' '.join(cmd)}")
    note("ok")


note(f"setup: source={SRC}")
run([sys.executable, "-m", "pip", "-q", "install", "-r", "requirements-train.txt"])


In [ ]:
# --- Eval-v1 + GPU preflight -------------------------------------------
import torch

run([sys.executable, "scripts/25_build_eval_v1.py"])
if not torch.cuda.is_available():
    raise RuntimeError("No GPU. Settings -> Accelerator -> GPU T4 x2.")
major, minor = torch.cuda.get_device_capability(0)
note(f"preflight: {torch.cuda.get_device_name(0)} sm_{major}{minor}; torch {torch.__version__}")
assert os.path.exists("outputs/eval-v1/base-768/predictions.jsonl"), "base-768 predictions missing from the snapshot"


In [ ]:
# --- the run: same reader, embed-v1 retriever ------------------------------
COMMON = ["--adapter", "none", "--split", "all", "--dense", "--k", "8",
          "--max-new-tokens", "768", "--batch-size", "2"]
run([sys.executable, "scripts/26_eval_v1_run.py", "--limit", "4", "--label", "smoke",
     "--dense-model", "NyayaLabs98/nyaya-embed-v1", *COMMON])
t0 = time.time()
run([sys.executable, "scripts/26_eval_v1_run.py", "--label", "base-768-embed-v1",
     "--dense-model", "NyayaLabs98/nyaya-embed-v1", *COMMON])
note(f"base-768-embed-v1 done in {(time.time() - t0) / 60:.0f} min")
run([sys.executable, "scripts/27_compare_runs.py", "--a", "base-768", "--b", "base-768-embed-v1"])


In [ ]:
# --- results: download these --------------------------------------------
import json, pathlib

out = pathlib.Path("/kaggle/working/nyaya-retriever-effect")
out.mkdir(exist_ok=True)
results = json.load(open("reports/eval_v1_results.json", encoding="utf-8"))
for label in ("base-768", "base-768-embed-v1"):
    m = results[label]["metrics"]
    note(f"{label:<20} fact_recall {m['fact_recall']:.1%}  citation {m['citation_accuracy']:.1%}  all_facts {m['all_facts_accuracy']:.1%}")
shutil.copy("reports/eval_v1_results.json", out)
shutil.copy("reports/eval_v1_comparison_base-768-embed-v1.json", out)
shutil.copy("outputs/eval-v1/base-768-embed-v1/predictions.jsonl", out / "base-768-embed-v1_predictions.jsonl")
shutil.make_archive("/kaggle/working/nyaya-retriever-effect", "zip", out)
note("collected: " + ", ".join(sorted(p.name for p in out.iterdir())))
